# Region classifier (spine / hip_left / hip_right) — обучение

Ноутбук для Kaggle/Colab (локально GPU нет). Обучает лёгкий CNN-классификатор
(3 класса: `spine`, `hip_left`, `hip_right`) на DXA-снимках поясничного отдела
позвоночника и проксимального отдела бедра — Шаг 1 плана (`docs/plan.md`).

## Перед запуском

Загрузите единый пакет, собранный `build_training_package.py`
(из исходной `Исследования` + `manifest.csv`), — плоская папка:

```
training_package/
  images/*.png   (252 PNG, имя файла = image_hash)
  manifest.csv   (колонка image_filename вместо вложенных путей)
```

Загрузите эту папку целиком как один Kaggle Dataset (или положите в
Google Drive для Colab) и пропишите `IMAGES_DIR`/`MANIFEST_PATH` в ячейке
конфигурации ниже. DICOM и pydicom ноутбуку больше не нужны — вся
конвертация уже сделана на этапе сборки пакета.

## Важно про метки

Метки `region`/`side` в `manifest.csv` — это не разметка организаторов
(per-image разметки зоны в датасете нет), а результат эвристического
OpenCV-классификатора (`services/region_classifier/classify.py`),
согласованный на 99.6% с независимой эвристикой по ширине снимка и
дополнительно проверенный на маленькой выборке с именами файлов
(3/3 совпадений) и ручным просмотром по классам (`export_for_review.py`,
найденные ошибки region/side исправлены в манифесте вручную).

## Воспроизводимость

Ноутбук фиксирует: seed для всех источников случайности (Python/NumPy/
PyTorch/CUDA/albumentations), детерминированные алгоритмы cuDNN, версии
всех ключевых библиотек (сохраняются в `run_info.json` вместе с
чекпоинтами), и хэш `manifest.csv`, на котором обучалась модель — чтобы
можно было убедиться, что при повторном запуске используются те же
данные и то же окружение. Полный bit-exact детерминизм на GPU не
гарантирован (некоторые CUDA-операции недетерминированы даже при
выставленных флагах — известное ограничение PyTorch, не баг этого
ноутбука), но воспроизводимость метрик с точностью до статистического
шума обеспечена.


In [ ]:
# Установка зависимостей. Версии пин-нутые для воспроизводимости —
# если Kaggle/Colab ругается на конфликт версий, ослабьте пины, но тогда
# обязательно зафиксируйте фактически установленные версии в run_info.json
# ниже (эта ячейка это делает автоматически после установки).
# pydicom не нужен — training_package уже содержит готовые PNG.
!pip install -q albumentations==2.0.8 timm==1.0.9 scikit-learn==1.5.2


In [ ]:
import os
import io
import json
import random
import hashlib
import platform
import subprocess
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import cv2
import albumentations as A
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix


## Воспроизводимость: seed + детерминизм

In [ ]:
SEED = 42


def set_full_determinism(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # cuDNN: детерминированные (но чуть более медленные) алгоритмы вместо
    # автотюнинга, который сам по себе недетерминирован между запусками.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_full_determinism(SEED)


def worker_init_fn(worker_id: int) -> None:
    # без этого воркеры DataLoader'а получают собственное недетерминированное
    # состояние RNG (albumentations внутри Dataset.__getitem__ использует
    # глобальный random/np.random, который в отдельном процессе-воркере
    # инициализируется по-своему без явного seed)
    worker_seed = SEED + worker_id
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def make_generator(seed: int = SEED) -> torch.Generator:
    g = torch.Generator()
    g.manual_seed(seed)
    return g


## Конфигурация путей и гиперпараметров

In [ ]:
# --- ПРАВЬТЕ ПОД СВОЮ СРЕДУ ---
# Kaggle пример (после загрузки training_package как Kaggle Dataset):
#   IMAGES_DIR = "/kaggle/input/dxa-training-package/images"
#   MANIFEST_PATH = "/kaggle/input/dxa-training-package/manifest.csv"
# Colab пример (после подключения Google Drive):
#   IMAGES_DIR = "/content/drive/MyDrive/dxa/training_package/images"
#   MANIFEST_PATH = "/content/drive/MyDrive/dxa/training_package/manifest.csv"

IMAGES_DIR = "/kaggle/input/datasets/cgvhbjmk/dxa-training-package/images"
MANIFEST_PATH = "/kaggle/input/datasets/cgvhbjmk/dxa-training-package/manifest.csv"
OUTPUT_DIR = "./outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_EPOCHS = 40
EARLY_STOPPING_PATIENCE = 7
HEAD_LR = 1e-3
BACKBONE_LR = 1e-5
N_FOLDS = 5  # должно совпадать с тем, что использовалось в build_manifest.py
TEST_FOLD = N_FOLDS - 1  # этот fold НИКОГДА не используется в обучении/CV —
# это настоящий отложенный тест, а не просто ещё один fold для валидации.
# Метки во всём manifest.csv (включая этот fold) вручную перепроверены по
# экспорту export_for_review.py, поэтому это честная human-verified оценка,
# а не просто сверка с прокси-эвристикой.
NUM_WORKERS = 2

# Аблация: отключить агрессивную геометрическую/шумовую аугментацию,
# чтобы проверить гипотезу, что именно она (в первую очередь CoarseDropout,
# закрывающий крыло подвздошной кости, и поворот/сдвиг) размывает тонкий
# сигнал стороны бедра и вызывает путаницу hip_left/hip_right.
# True  -> полная аугментация (как было)
# False -> только CLAHE + resize/pad, без искажений (как в eval)
ENABLE_TRAIN_AUGMENTATION = False  # эксперимент: проверяем гипотезу, что аугментация мешала стороне бедра
# Флип с меной label — отдельный тумблер: это не столько 'искажение',
# сколько способ удвоить обучающие примеры с ЗАВЕДОМО верной меткой
# (зеркальное бедро — валидный пример другой стороны). Обычно стоит
# оставлять True даже при ENABLE_TRAIN_AUGMENTATION=False.
ENABLE_FLIP_AUGMENTATION = True

# Backbone: ImageNet-претрейн через timm (скачивается автоматически при
# первом вызове timm.create_model(..., pretrained=True), кэшируется в
# ~/.cache/torch/hub). Опционально можно заменить на RadImageNet-веса
# (радиологический претрейн, см. https://github.com/BMEII-AI/RadImageNet) —
# для этого нужно вручную скачать .pt/.h5 веса по ссылке из репозитория,
# положить путь в RADIMAGENET_WEIGHTS_PATH и раскомментировать загрузку
# в build_model() ниже. По умолчанию используется ImageNet — самый
# надёжный вариант "из коробки" на Kaggle/Colab без внешних зависимостей.
RADIMAGENET_WEIGHTS_PATH = None  # например "/kaggle/input/radimagenet/resnet18.pt"

LABEL_TO_IDX = {"spine": 0, "hip_left": 1, "hip_right": 2}
IDX_TO_LABEL = {v: k for k, v in LABEL_TO_IDX.items()}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## Фиксация окружения (для воспроизводимости и отчётности)

In [ ]:
def file_md5(path: str) -> str:
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()


def get_pip_freeze() -> list[str]:
    try:
        out = subprocess.run(["pip", "freeze"], capture_output=True, text=True, check=True)
        return out.stdout.splitlines()
    except Exception as exc:  # noqa: BLE001
        return [f"pip freeze failed: {exc}"]


run_info = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "timm_version": timm.__version__,
    "albumentations_version": A.__version__,
    "manifest_path": MANIFEST_PATH,
    "manifest_md5": file_md5(MANIFEST_PATH),
    "hyperparameters": {
        "image_size": IMAGE_SIZE,
        "batch_size": BATCH_SIZE,
        "num_epochs": NUM_EPOCHS,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "head_lr": HEAD_LR,
        "backbone_lr": BACKBONE_LR,
        "n_folds": N_FOLDS,
        "backbone": "resnet18 (timm, ImageNet pretrained)" if RADIMAGENET_WEIGHTS_PATH is None else f"resnet18 (RadImageNet: {RADIMAGENET_WEIGHTS_PATH})",
        "enable_train_augmentation": ENABLE_TRAIN_AUGMENTATION,
        "enable_flip_augmentation": ENABLE_FLIP_AUGMENTATION,
    },
    "label_to_idx": LABEL_TO_IDX,
}

with open(os.path.join(OUTPUT_DIR, "run_info.json"), "w", encoding="utf-8") as f:
    json.dump(run_info, f, ensure_ascii=False, indent=2)
with open(os.path.join(OUTPUT_DIR, "pip_freeze.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(get_pip_freeze()))

print(json.dumps(run_info, ensure_ascii=False, indent=2))


## Аугментация

Зеркалит `services/region_classifier/training/augmentation.py` — держим
копию прямо в ноутбуке, чтобы не тащить весь репозиторий в Kaggle/Colab
(ноутбук самодостаточен, нужен только manifest.csv + сами DICOM).

Напоминание по дизайну (подробнее — в докстринге исходного файла):
- горизонтальный флип обрабатывается отдельно и меняет метку `hip_left <-> hip_right`
- НЕ используется вертикальный флип и повороты >12° (угол наклона —
  часть отдельной будущей задачи контроля качества, не должен здесь
  "выучиваться как несущественный")
- контраст/гамма/шум аугментируются заметно сильнее обычного — суррогат
  разброса между разными DXA-аппаратами (Hologic/iDXA), которых в
  обучающих данных физически нет, а организаторы просили generalization


In [ ]:
FLIP_LABEL_SWAP = {"hip_left": "hip_right", "hip_right": "hip_left"}


def flip_with_label_swap(image: np.ndarray, label: str, p: float = 0.5):
    if random.random() >= p:
        return image, label
    flipped = cv2.flip(image, 1)
    return flipped, FLIP_LABEL_SWAP.get(label, label)


def clahe(image: np.ndarray) -> np.ndarray:
    return cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8)).apply(image)


def build_train_transform(image_size=IMAGE_SIZE) -> A.Compose:
    return A.Compose([
        A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.8),
        A.RandomGamma(gamma_limit=(70, 130), p=0.5),
        A.OneOf([
            A.GaussNoise(std_range=(0.02, 0.08), p=1.0),
            A.MultiplicativeNoise(multiplier=(0.9, 1.1), p=1.0),
        ], p=0.4),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
            A.Sharpen(alpha=(0.1, 0.3), p=1.0),
        ], p=0.3),
        A.Affine(scale=(0.92, 1.08), translate_percent=(0.0, 0.06), rotate=(-12, 12), fit_output=False, p=0.7),
        A.CoarseDropout(num_holes_range=(1, 2), hole_height_range=(0.05, 0.18),
                         hole_width_range=(0.05, 0.18), fill=0, p=0.25),
        A.LongestMaxSize(max_size=image_size),
        A.PadIfNeeded(min_height=image_size, min_width=image_size, border_mode=cv2.BORDER_CONSTANT, fill=0),
    ], seed=SEED)


def build_eval_transform(image_size=IMAGE_SIZE) -> A.Compose:
    return A.Compose([
        A.LongestMaxSize(max_size=image_size),
        A.PadIfNeeded(min_height=image_size, min_width=image_size, border_mode=cv2.BORDER_CONSTANT, fill=0),
    ])


def augment_sample(image_u8: np.ndarray, label: str, train: bool, image_size=IMAGE_SIZE):
    image = clahe(image_u8)
    if train:
        if ENABLE_FLIP_AUGMENTATION:
            image, label = flip_with_label_swap(image, label)
        transform = build_train_transform(image_size) if ENABLE_TRAIN_AUGMENTATION else build_eval_transform(image_size)
    else:
        transform = build_eval_transform(image_size)
    augmented = transform(image=image)["image"]
    return augmented, label


## Dataset и загрузка манифеста

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


class RegionDataset(Dataset):
    # train=True -> аугментация + случайный flip с меной метки.
    # train=False -> детерминированный eval-препроцессинг.

    def __init__(self, df: pd.DataFrame, images_dir: str, train: bool):
        self.df = df.reset_index(drop=True)
        self.images_dir = images_dir
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.images_dir, row["image_filename"])
        image_u8 = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if image_u8 is None:
            raise FileNotFoundError(f"Не удалось прочитать {path}")
        image_u8, label = augment_sample(image_u8, row["label"], train=self.train)

        # grayscale -> 3 канала (претрейн ждёт RGB) + ImageNet-нормализация
        rgb = np.stack([image_u8] * 3, axis=-1).astype(np.float32) / 255.0
        rgb = (rgb - IMAGENET_MEAN) / IMAGENET_STD
        tensor = torch.from_numpy(rgb.transpose(2, 0, 1)).float()

        return tensor, LABEL_TO_IDX[label]


manifest = pd.read_csv(MANIFEST_PATH)
print("Всего строк в манифесте:", len(manifest))
print(manifest["label"].value_counts())
print(manifest["fold"].value_counts().sort_index())

# Разделяем на CV-часть (для обучения и подбора модели) и настоящий
# отложенный тест (fold == TEST_FOLD) — эти данные не участвуют ни в
# обучении, ни в выборе лучшей эпохи ни для одного фолда, видим их
# только один раз, в самом конце.
cv_manifest = manifest[manifest["fold"] != TEST_FOLD].reset_index(drop=True)
test_manifest = manifest[manifest["fold"] == TEST_FOLD].reset_index(drop=True)
print(f"\nCV (обучение/валидация): {len(cv_manifest)} изображений")
print(cv_manifest["label"].value_counts())
print(f"\nОтложенный тест (fold {TEST_FOLD}, не используется в обучении): {len(test_manifest)} изображений")
print(test_manifest["label"].value_counts())


## Модель: ResNet18 с ImageNet-претрейном, частичная заморозка

In [ ]:
def build_model(num_classes: int = 3) -> nn.Module:
    model = timm.create_model("resnet18", pretrained=True, num_classes=num_classes)

    if RADIMAGENET_WEIGHTS_PATH is not None:
        state = torch.load(RADIMAGENET_WEIGHTS_PATH, map_location="cpu", weights_only=False)
        missing, unexpected = model.load_state_dict(state, strict=False)
        print("RadImageNet веса загружены. missing:", missing, "unexpected:", unexpected)

    # Замораживаем ранние слои (общие низкоуровневые фильтры не нужно
    # переучивать на ~250 изображениях) — оставляем обучаемыми layer3, layer4
    # и голову. См. обоснование в docs/plan.md, Шаг 1.
    for name, param in model.named_parameters():
        if name.startswith(("conv1", "bn1", "layer1", "layer2")):
            param.requires_grad = False

    return model


def build_optimizer(model: nn.Module):
    head_params = []
    backbone_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith("fc"):
            head_params.append(param)
        else:
            backbone_params.append(param)
    return torch.optim.AdamW([
        {"params": head_params, "lr": HEAD_LR},
        {"params": backbone_params, "lr": BACKBONE_LR},
    ])


## Цикл обучения/валидации на одном фолде

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    all_preds, all_labels = [], []

    with torch.set_grad_enabled(is_train):
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, acc, f1_macro, all_preds, all_labels


def train_one_fold(fold: int, manifest: pd.DataFrame):
    # Пересеиваем перед каждым фолдом — иначе порядок фолдов влиял бы на
    # то, какое RNG-состояние достаётся каждому, и повторный запуск с
    # другим N_FOLDS/порядком дал бы другой результат для того же фолда.
    set_full_determinism(int(SEED + fold))

    train_df = manifest[manifest["fold"] != fold]
    val_df = manifest[manifest["fold"] == fold]

    train_ds = RegionDataset(train_df, IMAGES_DIR, train=True)
    val_ds = RegionDataset(val_df, IMAGES_DIR, train=False)
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
        drop_last=True, worker_init_fn=worker_init_fn, generator=make_generator(int(SEED + fold)),
    )
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    model = build_model().to(DEVICE)
    optimizer = build_optimizer(model)
    criterion = nn.CrossEntropyLoss()

    best_val_f1 = -1.0
    best_state = None
    epochs_without_improvement = 0
    history = []

    for epoch in range(NUM_EPOCHS):
        train_loss, train_acc, train_f1, _, _ = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, val_f1, val_preds, val_labels = run_epoch(model, val_loader, criterion)

        history.append(dict(
            epoch=epoch + 1, train_loss=train_loss, train_acc=train_acc, train_f1=train_f1,
            val_loss=val_loss, val_acc=val_acc, val_f1=val_f1,
        ))
        print(f"fold {fold} | epoch {epoch+1}/{NUM_EPOCHS} | "
              f"train_loss={train_loss:.4f} train_f1={train_f1:.4f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_preds, best_labels = val_preds, val_labels
            best_epoch = epoch + 1
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                print(f"fold {fold}: early stopping на эпохе {epoch+1}")
                break

    # Чекпоинт сохраняем вместе с конфигом/метаданными — не только веса,
    # чтобы через месяц можно было понять, чем именно это обучено, без
    # необходимости листать историю ноутбука.
    checkpoint = {
        "model_state_dict": best_state,
        "label_to_idx": LABEL_TO_IDX,
        "image_size": IMAGE_SIZE,
        "best_epoch": best_epoch,
        "best_val_f1": best_val_f1,
        "fold": fold,
        "run_info": run_info,
    }
    torch.save(checkpoint, os.path.join(OUTPUT_DIR, f"region_classifier_fold{fold}.pt"))

    pd.DataFrame(history).to_csv(os.path.join(OUTPUT_DIR, f"history_fold{fold}.csv"), index=False)

    return best_val_f1, best_preds, best_labels


## Запуск 5-fold кросс-валидации

In [ ]:
fold_scores = []
all_val_preds, all_val_labels = [], []

for fold in sorted(int(f) for f in cv_manifest["fold"].unique()):
    best_f1, preds, labels = train_one_fold(fold, cv_manifest)
    fold_scores.append(best_f1)
    all_val_preds.extend(preds)
    all_val_labels.extend(labels)
    print(f"=== fold {fold}: best val macro-F1 = {best_f1:.4f} ===\n")

print("Macro-F1 по фолдам:", [round(s, 4) for s in fold_scores])
print("Среднее:", round(np.mean(fold_scores), 4), "± std:", round(np.std(fold_scores), 4))

with open(os.path.join(OUTPUT_DIR, "cv_summary.json"), "w", encoding="utf-8") as f:
    json.dump({
        "fold_scores": fold_scores,
        "mean_f1": float(np.mean(fold_scores)),
        "std_f1": float(np.std(fold_scores)),
    }, f, ensure_ascii=False, indent=2)


## Метрики по CV (folds 0..TEST_FOLD-1) — для выбора модели, не финальная оценка

Это честная out-of-fold оценка, но только на CV-части данных (без последнего, отложенного fold'а). Настоящая финальная метрика — на `test_manifest` (последний fold, TEST_FOLD), которая ни разу не участвовала ни в обучении, ни в подборе лучшей эпохи — см. следующий раздел.

In [ ]:
target_names = [IDX_TO_LABEL[i] for i in range(len(IDX_TO_LABEL))]
report_text = classification_report(all_val_labels, all_val_preds, target_names=target_names, digits=3)
print(report_text)

cm = confusion_matrix(all_val_labels, all_val_preds)
cm_df = pd.DataFrame(cm, index=target_names, columns=target_names)
print("Confusion matrix (строки — истинные, столбцы — предсказанные):")
print(cm_df)

with open(os.path.join(OUTPUT_DIR, "classification_report.txt"), "w", encoding="utf-8") as f:
    f.write(report_text)
cm_df.to_csv(os.path.join(OUTPUT_DIR, "confusion_matrix.csv"))


## Ансамбль из фолдов CV (folds 0..TEST_FOLD-1)

Ансамбль собирается из моделей, обученных в CV-цикле выше (каждая видела все CV-данные, кроме своего собственного validation-фолда, и НИ ОДНА не видела `test_manifest` — тот самый последний, отложенный fold). Поэтому усреднять их предсказания на `test_manifest` — честно, без утечки (в отличие от повторного использования CV-фолдов друг для друга: там каждая CV-модель видела чужие CV-фолды при обучении, но не тестовый). Собственно эта честная проверка — в следующем разделе.

In [ ]:
class EnsembleClassifier:
    def __init__(self, checkpoint_paths: list[str], device: torch.device = DEVICE):
        self.device = device
        self.models = []
        self.label_to_idx = None
        self.image_size = IMAGE_SIZE

        for path in checkpoint_paths:
            ckpt = torch.load(path, map_location=device, weights_only=False)
            if self.label_to_idx is None:
                self.label_to_idx = ckpt["label_to_idx"]
                self.image_size = ckpt["image_size"]
            elif ckpt["label_to_idx"] != self.label_to_idx:
                raise ValueError(f"{path}: label_to_idx не совпадает с остальными чекпоинтами фолдов")

            model = build_model(num_classes=len(ckpt["label_to_idx"]))
            model.load_state_dict(ckpt["model_state_dict"])
            model.to(device).eval()
            self.models.append(model)

        self.idx_to_label = {v: k for k, v in self.label_to_idx.items()}
        print(f"Загружено моделей в ансамбль: {len(self.models)}")

    def _preprocess(self, image_u8: np.ndarray) -> torch.Tensor:
        # тот же детерминированный eval-препроцессинг, что и на валидации:
        # CLAHE -> resize/pad, БЕЗ случайной аугментации (train=False)
        image, _ = augment_sample(image_u8, label="spine", train=False, image_size=self.image_size)
        rgb = np.stack([image] * 3, axis=-1).astype(np.float32) / 255.0
        rgb = (rgb - IMAGENET_MEAN) / IMAGENET_STD
        tensor = torch.from_numpy(rgb.transpose(2, 0, 1)).float()
        return tensor.unsqueeze(0).to(self.device)

    @torch.no_grad()
    def predict(self, image_u8: np.ndarray) -> dict:
        # image_u8: grayscale uint8 numpy array (как из cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        # или из load_pixel_array для сырого DICOM — препроцессинг тот же).
        tensor = self._preprocess(image_u8)

        per_model_probs = []
        for model in self.models:
            logits = model(tensor)
            probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
            per_model_probs.append(probs)

        per_model_probs = np.stack(per_model_probs, axis=0)  # (n_models, n_classes)
        avg_probs = per_model_probs.mean(axis=0)
        pred_idx = int(avg_probs.argmax())

        return {
            "label": self.idx_to_label[pred_idx],
            "confidence": float(avg_probs[pred_idx]),
            "probs_by_class": {self.idx_to_label[i]: float(p) for i, p in enumerate(avg_probs)},
            "per_model_agreement": float((per_model_probs.argmax(axis=1) == pred_idx).mean()),
        }


# Пример использования на новом снимке (замените путь на реальный):
#
# ensemble = EnsembleClassifier([
#     os.path.join(OUTPUT_DIR, f"region_classifier_fold{i}.pt") for i in range(N_FOLDS)
# ])
# new_image = cv2.imread("/path/to/new_image.png", cv2.IMREAD_GRAYSCALE)
# result = ensemble.predict(new_image)
# print(result)
#
# per_model_agreement < 1.0 — сигнал разногласия моделей между собой,
# полезно как дополнительная мера неуверенности (аналогично
# region_confidence/side_confidence в classify.py) — если модели
# ансамбля расходятся, стоит пометить снимок на ручную проверку.

## Финальная оценка ансамбля на настоящем отложенном тесте (внутри наших данных)

Это главная метрика, а не CV-оценка из раздела выше. `test_manifest` (fold == TEST_FOLD, ~1/5 данных) не участвовала НИ В ОДНОМ из 4 обучающих прогонов выше — ни в обучении, ни в выборе лучшей эпохи ни для одного фолда. Метки в ней — та же вручную перепроверенная разметка, что и во всём датасете (не отдельная процедура).

In [ ]:
ensemble = EnsembleClassifier([
    os.path.join(OUTPUT_DIR, f"region_classifier_fold{fold}.pt")
    for fold in sorted(int(f) for f in cv_manifest["fold"].unique())
])

test_preds, test_labels_true, test_confidences, test_agreements = [], [], [], []
for _, row in test_manifest.iterrows():
    img = cv2.imread(os.path.join(IMAGES_DIR, row["image_filename"]), cv2.IMREAD_GRAYSCALE)
    result = ensemble.predict(img)
    test_preds.append(LABEL_TO_IDX[result["label"]])
    test_labels_true.append(LABEL_TO_IDX[row["label"]])
    test_confidences.append(result["confidence"])
    test_agreements.append(result["per_model_agreement"])

test_report = classification_report(test_labels_true, test_preds, target_names=target_names, digits=3)
print(f"Отложенный тест: {len(test_manifest)} изображений (никогда не участвовали в обучении)")
print(test_report)

test_cm = confusion_matrix(test_labels_true, test_preds)
test_cm_df = pd.DataFrame(test_cm, index=target_names, columns=target_names)
print("Confusion matrix (отложенный тест):")
print(test_cm_df)
print(f"\nСредняя уверенность ансамбля: {np.mean(test_confidences):.3f}, "
      f"среднее согласие моделей между собой: {np.mean(test_agreements):.3f}")

with open(os.path.join(OUTPUT_DIR, "holdout_test_classification_report.txt"), "w", encoding="utf-8") as f:
    f.write(test_report)
test_cm_df.to_csv(os.path.join(OUTPUT_DIR, "holdout_test_confusion_matrix.csv"))


## Дополнительная проверка: внешний скан-сеанс ("Для теста", n=3)

Основная финальная метрика — уже выше, на 50 отложенных изображениях из нашего датасета. Этот раздел — дополнительный, не главный, sanity-check: 3 файла из отдельной папки организаторов "Для теста", снятые в другой скан-сессии (не входят в наш датасет вообще). Метки здесь — из имени файла (`_ПОП`/`_ППОБ`/`_ЛПОБ`), не из нашей ручной проверки. Полезно как ещё один независимый источник, но n=3 всё равно мало для статистики — смотрим на это как на дополнительное подтверждение, а не отдельную метрику.

In [ ]:
# --- ПРАВЬТЕ ПОД СВОЮ СРЕДУ ---
HOLDOUT_IMAGES_DIR = "/kaggle/input/holdout-test-package/images"
HOLDOUT_MANIFEST_PATH = "/kaggle/input/holdout-test-package/manifest_holdout.csv"

ensemble = EnsembleClassifier([
    os.path.join(OUTPUT_DIR, f"region_classifier_fold{i}.pt") for i in range(N_FOLDS)
])

holdout = pd.read_csv(HOLDOUT_MANIFEST_PATH)
correct = 0
for _, row in holdout.iterrows():
    img = cv2.imread(os.path.join(HOLDOUT_IMAGES_DIR, row["image_filename"]), cv2.IMREAD_GRAYSCALE)
    result = ensemble.predict(img)
    is_correct = result["label"] == row["true_label"]
    correct += is_correct
    print(f"{row['source_filename']}: true={row['true_label']} pred={result['label']} "
          f"(conf={result['confidence']:.3f}, agreement={result['per_model_agreement']:.2f}) "
          f"{'OK' if is_correct else 'MISMATCH'}")

print(f"\n{correct}/{len(holdout)} совпадений с истинной разметкой (n слишком мал для статистики)")


## Что дальше

- В `./outputs/` сохранено всё для воспроизводимости и аудита:
  `region_classifier_fold{N}.pt` (веса + конфиг + label_to_idx + run_info
  внутри одного файла), `history_fold{N}.csv` (метрики по эпохам),
  `run_info.json` (версии библиотек, seed, hash манифеста, гиперпараметры),
  `pip_freeze.txt` (полный список версий пакетов), `cv_summary.json`,
  `classification_report.txt`, `confusion_matrix.csv`.
- Для инференса можно усреднить предсказания всех 5 фолдов (ensemble) или
  взять фолд с лучшим val-F1 (см. `best_val_f1` внутри чекпоинта).
- Если результат не превосходит OpenCV-фолбэк (99.6% на прокси-оценке,
  см. `services/region_classifier/README.md`) — это ожидаемо на таком
  маленьком датасете; ценность этой модели — в потенциальной
  генерализации на снимки других DXA-аппаратов за счёт агрессивной
  контрастной аугментации, которую нельзя оценить без реальных
  multi-vendor данных (см. обсуждение датасетов UK Biobank / VerteNet /
  Bone Densitometry Dataset в истории работы над проектом).
- Следующий шаг по плану (`docs/plan.md`) — Шаг 2: keypoint-модель для
  критериев укладки/оси/ROI.
